## Features

The models are trained, cross validated and compared on 4 feature sets.

**Feature Set 1**: only player statistical features

**Feature Set 2**: statistical features based on player statistic differences. Intuitively, one can argue that differences in player statistics—such as match win ratio or frame win ratio—may be sufficient for the prediction scenario that we have.

**Feature Set 3**: player elo ratings and features based on player statistic differences

**Feature Set 4**: all available features

## Model Selection

**KPI**: weighted average root mean squared error (WRMSE) and root mean squared error (RMSE)

We divide our data into six chronologically ordered windows, reserving the last chunk as a holdout set for final model evaluation. 

For cross-validation during model training and hyperparameter tuning, we use the first five chunks with time series splitting. To compare model performance during development, we employ a metric based on mean squared error and training size, weighted average root mean squared error (WRMSE), across all folds. Since the training data sizes vary across the time series split folds, using a weighted average—weighted by the training sizes—provides a fairer comparison.

RMSE on the holdout set is used to evaluate final performance of the models.

## Comparison

RMSE for the baseline model was 0.31230869425101954.

**Elo** CV WRMSE : 0.2714

**Elo** final holdout set RMSE : 0.2666

All models except linear regression are hyperparameter tuned using Grid Search CV. Best tuned models will be used for performance comparison on final holdout set.

- **Linear Regression**
  - Feature set 1 CV WRMSE : 0.2809
  - Feature set 2 CV WRMSE : 0.2815
  - Feature set 3 CV WRMSE : 0.2710
  - Feature set 4 CV WRMSE : 0.2734
- **kNN**
  - Feature set 1 CV WRMSE : 0.2919
  - Feature set 2 CV WRMSE : 0.2939
  - Feature set 3 CV WRMSE : 0.2843
  - Feature set 4 CV WRMSE : 0.2904
- **Decision Tree**
  - Feature set 1 CV WRMSE : 0.2868
  - Feature set 2 CV WRMSE : 0.2825
  - Feature set 3 CV WRMSE : 0.2833
  - Feature set 4 CV WRMSE : 0.2757
- **Random Forest**
  - Feature set 1 CV WRMSE : 0.2755
  - Feature set 2 CV WRMSE : 0.2814
  - Feature set 3 CV WRMSE : 0.2741
  - Feature set 4 CV WRMSE : 0.2707
- **XGBoost**
  - Feature set 1 CV WRMSE : 0.2762
  - Feature set 2 CV WRMSE : 0.2802
  - Feature set 3 CV WRMSE : 0.2742
  - Feature set 4 CV WRMSE : 0.2715

  Based on these, we choose Linear Regression, Random Forest and XGBoost to test on the holdout set.




## Testing on holdout

In [1]:
import matplotlib.pyplot as plt 
import numpy as np 
import os 
import pandas as pd 

In [2]:
df = pd.read_csv('match_data_50_tourns_modified.csv')
n = round(len(df) / 6)
df_holdout = df.tail(n)
df=df.iloc[:-n]

In [3]:
#features set 1
features_to_remove = ['player1', 'player2', 'player1_elo','player2_elo','elo_match_win_rate','elo_frame_win_rate',
                      'score1','score2','match_result','win_percentage','tournament_id']
X1 = df.drop(columns=features_to_remove)
y1 = df['win_percentage']
X1_test = df_holdout.drop(columns=features_to_remove)
y1_test = df_holdout['win_percentage']

In [4]:
#feature set 2
dfm = df
dfm['p1_matches_win_ratio']=dfm['p1_matches_won']/df['p1_matches_played']
dfm['p2_matches_win_ratio']=dfm['p2_matches_won']/df['p2_matches_played']
dfm['p1_frames_win_ratio']=dfm['p1_frames_won']/df['p1_frames_played']
dfm['p2_frames_win_ratio']=dfm['p2_frames_won']/df['p2_frames_played']
dfm['p1_frames_win_ratio_1_year']=dfm['p1_frames_won_1_year']/df['p1_frames_played_1_year']
dfm['p2_frames_win_ratio_1_year']=dfm['p2_frames_won_1_year']/df['p2_frames_played_1_year']
dfm['p1_frames_win_ratio_3_years']=dfm['p1_frames_won_3_years']/df['p1_frames_played_3_years']
dfm['p2_frames_win_ratio_3_years']=dfm['p2_frames_won_3_years']/df['p2_frames_played_3_years']
dfm.fillna(0.5, inplace=True)
dfm['matches_win_ratio_diff']=dfm['p1_matches_win_ratio']-dfm['p2_matches_win_ratio']
dfm['frames_win_ratio_diff']=dfm['p1_frames_win_ratio']-dfm['p2_frames_win_ratio']
dfm['frames_win_ratio_diff_1_year']=dfm['p1_frames_win_ratio_1_year']-dfm['p2_frames_win_ratio_1_year']
dfm['frames_win_ratio_diff_3_years']=dfm['p1_frames_win_ratio_3_years']-dfm['p2_frames_win_ratio_3_years']
selected_features = ['matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']
X2 = dfm[selected_features]
y2 = dfm['win_percentage']

In [5]:
#feature set 3
selected_features = ['player1_elo','player2_elo','matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']
X3 = dfm[selected_features]
y3 = dfm['win_percentage']

In [6]:
dfm = df_holdout
dfm['p1_matches_win_ratio']=dfm['p1_matches_won']/df['p1_matches_played']
dfm['p2_matches_win_ratio']=dfm['p2_matches_won']/df['p2_matches_played']
dfm['p1_frames_win_ratio']=dfm['p1_frames_won']/df['p1_frames_played']
dfm['p2_frames_win_ratio']=dfm['p2_frames_won']/df['p2_frames_played']
dfm['p1_frames_win_ratio_1_year']=dfm['p1_frames_won_1_year']/df['p1_frames_played_1_year']
dfm['p2_frames_win_ratio_1_year']=dfm['p2_frames_won_1_year']/df['p2_frames_played_1_year']
dfm['p1_frames_win_ratio_3_years']=dfm['p1_frames_won_3_years']/df['p1_frames_played_3_years']
dfm['p2_frames_win_ratio_3_years']=dfm['p2_frames_won_3_years']/df['p2_frames_played_3_years']
dfm.fillna(0.5, inplace=True)
dfm['matches_win_ratio_diff']=dfm['p1_matches_win_ratio']-dfm['p2_matches_win_ratio']
dfm['frames_win_ratio_diff']=dfm['p1_frames_win_ratio']-dfm['p2_frames_win_ratio']
dfm['frames_win_ratio_diff_1_year']=dfm['p1_frames_win_ratio_1_year']-dfm['p2_frames_win_ratio_1_year']
dfm['frames_win_ratio_diff_3_years']=dfm['p1_frames_win_ratio_3_years']-dfm['p2_frames_win_ratio_3_years']
selected_features = ['matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']
X2_test = dfm[selected_features]
y2_test = dfm['win_percentage']
selected_features = ['player1_elo','player2_elo','matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']
X3_test = dfm[selected_features]
y3_test = dfm['win_percentage']

In [7]:
#feature set 4
features_to_remove = ['player1', 'player2', 
                      'score1','score2','match_result','win_percentage','tournament_id']
X4 = df.drop(columns=features_to_remove) 
y4 = df['win_percentage']
X4_test = df_holdout.drop(columns=features_to_remove) 
y4_test = df_holdout['win_percentage']

### Linear Regression

In [8]:
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X1, y1)

y_pred = model.predict(X1_test)

rmse = root_mean_squared_error(y1_test, y_pred)
print(f"Test RMSE with feature set 1: {rmse} \n")

Test RMSE with feature set 1: 0.28001722138652724 



In [9]:
model = LinearRegression()
model.fit(X2, y2)

y_pred = model.predict(X2_test)

rmse = root_mean_squared_error(y2_test, y_pred)
print(f"Test RMSE with feature set 2: {rmse} \n")

Test RMSE with feature set 2: 0.2915443930960063 



In [10]:
model = LinearRegression()
model.fit(X3, y3)

y_pred = model.predict(X3_test)

rmse = root_mean_squared_error(y3_test, y_pred)
print(f"Test RMSE with feature set 3: {rmse} \n")

Test RMSE with feature set 3: 0.26972518115006955 



In [11]:
model = LinearRegression()
model.fit(X4, y4)

y_pred = model.predict(X4_test)

rmse = root_mean_squared_error(y4_test, y_pred)
print(f"Test RMSE with feature set 4: {rmse} \n")

Test RMSE with feature set 4: 0.26835816899022064 



### Random Forest

In [12]:
from sklearn.ensemble import RandomForestRegressor

params = {
    'max_depth': 5,
    'min_samples_leaf': 4,
    'min_samples_split': 2,
    'n_estimators': 100
}

model = RandomForestRegressor(**params)
model.fit(X1, y1)

y_pred = model.predict(X1_test)

rmse = root_mean_squared_error(y1_test, y_pred)
print(f"Test RMSE with feature set 1: {rmse} \n")

Test RMSE with feature set 1: 0.2647552312393261 



In [13]:
params = {
    'max_depth': 5,
    'min_samples_leaf': 1,
    'min_samples_split': 5,
    'n_estimators': 200
}

model = RandomForestRegressor(**params)
model.fit(X2, y2)

y_pred = model.predict(X2_test)

rmse = root_mean_squared_error(y2_test, y_pred)
print(f"Test RMSE with feature set 2: {rmse} \n")


Test RMSE with feature set 2: 0.2909335370225883 



In [14]:
params = {
    'max_depth': 5,
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 50
}

model = RandomForestRegressor(**params)
model.fit(X3, y3)

y_pred = model.predict(X3_test)

rmse = root_mean_squared_error(y3_test, y_pred)
print(f"Test RMSE with feature set 3: {rmse} \n")

Test RMSE with feature set 3: 0.2704238390898377 



In [15]:
params = {
    'max_depth': 5,
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 200
}

model = RandomForestRegressor(**params)
model.fit(X4, y4)

y_pred = model.predict(X4_test)

rmse = root_mean_squared_error(y4_test, y_pred)
print(f"Test RMSE with feature set 4: {rmse} \n")

Test RMSE with feature set 4: 0.25954103914805604 



### XG Boost

In [16]:
from xgboost import XGBRegressor

params = {
    'colsample_bytree': 0.8,
    'learning_rate': 0.01,
    'max_depth': 5,
    'n_estimators': 200,
    'subsample': 0.8
}

model = XGBRegressor(**params)
model.fit(X1, y1)

y_pred = model.predict(X1_test)

rmse = root_mean_squared_error(y1_test, y_pred)
print(f"Test RMSE with feature set 1: {rmse} \n")

Test RMSE with feature set 1: 0.264739515816249 



In [17]:
params = {
    'colsample_bytree': 0.8,
    'learning_rate': 0.01,
    'max_depth': 3,
    'n_estimators': 200,
    'subsample': 0.8
}

model = XGBRegressor(**params)
model.fit(X2, y2)

y_pred = model.predict(X2_test)

rmse = root_mean_squared_error(y2_test, y_pred)
print(f"Test RMSE with feature set 2: {rmse} \n")

Test RMSE with feature set 2: 0.2920073250185874 



In [18]:
params = {
    'colsample_bytree': 0.8,
    'learning_rate': 0.1,
    'max_depth': 3,
    'n_estimators': 100,
    'subsample': 1.0
}

model = XGBRegressor(**params)
model.fit(X3, y3)

y_pred = model.predict(X3_test)

rmse = root_mean_squared_error(y3_test, y_pred)
print(f"Test RMSE with feature set 3: {rmse} \n")

Test RMSE with feature set 3: 0.2648485675351001 



In [19]:
params = {
    'colsample_bytree': 0.8,
    'learning_rate': 0.01,
    'max_depth': 3,
    'n_estimators': 200,
    'subsample': 0.8
}

model = XGBRegressor(**params)
model.fit(X4, y4)

y_pred = model.predict(X4_test)

rmse = root_mean_squared_error(y4_test, y_pred)
print(f"Test RMSE with feature set 4: {rmse} \n")

Test RMSE with feature set 4: 0.26075452517994224 



- **Linear Regression**
  - Test RMSE using feature set 1: 0.2800
  - Test RMSE using feature set 2: 0.2915
  - Test RMSE using feature set 3: 0.2697
  - Test RMSE using feature set 4: 0.2684
- **Random Forest**
  - Test RMSE using feature set 1: 0.2648
  - Test RMSE using feature set 2: 0.2909
  - Test RMSE using feature set 3: 0.2704
  - Test RMSE using feature set 4: 0.2595
- **XGBoost**
  - Test RMSE using feature set 1: 0.2647
  - Test RMSE using feature set 2: 0.2920
  - Test RMSE using feature set 3: 0.2648
  - Test RMSE using feature set 4: 0.2608